DEPLOY DI MODELLI DI TENSORFLOW (SAVEDMODEL->TFSERVING)

Il Deploy è il passaggio da 'modello allenato sul mio pc' a 'servizio chiamabile da altri programmi'
SaveModel non è un singolo file, è una cartela che contiene il modello, i pesi, le funzioni di inferenza e le firme input/output. 
Esempio semplice:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")

# dopo il training
model.export("modelli/peso_articolo/1")

1 identifica la versione del modelli, se domani si allena un modello migliore lo si salva con 2, 3, ecc

Per portare i modelli dal laboratoria al mondo reale.

- Ecosistemi di produzione e scallabilità con TensorFlow serving
- Deep Learning on-device con TF Lite
- Ingegnerizzazione del ciclo di vita: monitoraggio e manutenzione in produzione

Immagina di aver addestrato un modello.
Non è sufficiente caricare un file in uno script Python; serve un'infrastruttura capace di gestire migliaia di richiesta simultane.
TenrsorFlow Serving è il sistema di riferimento per esporre modelli tramite API REST o gRPC. Permette di gestire il versionamento dei modelli e l'aggiornamento a caldo senza interruzioni del servizio.

Il Model Server isola la complessità del modello dalla logica del sito o della APP. Ha la gestione delle versioni.


Il batching Dinamico di TF Serving può raggruppare richiede individuali in un unico batch per sfruttare meglio il parallelismo della GPU. Per applicazioni a bassissima latenza supporta gRPC

Il processo inizia con l'esportazione del mo dello nel formato 'SavedModel'. TF Serving legge questa directory e istanzia istantaneamente l'interfaccia di predizione.
Questo approccio permette una separazione netta dei ruoli tra chi addestra il modello (Data Scientist) e chi gestisce l'infrastruttura (ML Engineer).
Dalla creatività artigianale all'efficienza di una catena di montaggio industriale

E se il modlelo non deve stare sul server ma in tasca all'utente.
Magari per privacy non vogliamo mettere le foto sul server.
Oppure la letenza, un auto a guida autonoma non può aspettare le risposte dal server mentra sta frenando.
TensorFlow Lite porta l'intelligenza direttamente sul smartphon o sensori minuscoli.

Miniaturizzare significa ottimizzare.
1) La conversione trasforma il modello in formato 'FlagBuffer', riducendo le dimensioni del file rispetto ai pesi originali.
2) La 'Quantizzazione' riduce la precisione del pesi da float a 8 bit, diminuendo l'occupazione di memoria e accelerendo i calcoli. Riduciamo il peso del modello dell'80%
3) L'interprete TFLite è progettato per caricare solo le operazioni necessarie, riducendo l'overhead energetico sui dispositivi a batteria.
Il processo di quantizzazione lineare mappa i valori reali in un intervallo discreto di interi.

Il protagonista è TFLite Converter.
E' lo strumento Python che analizza il grafo di TensorFlow e lo traduce/semplifica in una sequenza di operazioni compatibili con l'interprete mobile.
Inoltre possiamo sfruttare i Delegati Hardware, se lo Smarphone ha un chip dedicato all'intelligenza artificiale (GPU dello smartphone o le NPU), TF Lite può parlare direttamente con quel chip bypassando la CPU generica 


Ovviamente comprimere un modello non è privo di sfide.
C'è sempre un compromesso, comprimere troppo può far perdere un po' di lucidità nel modello, il compito dell'architetto è trovare il giusto compromesso.
Il Deep Learnign smette di essere solo matematica e diventa ingenerie del progetto, dove le risorse harware fanno da padrone.

Ciclo di vita
Degrado delle prestazioni
Il lavoro non finisce con il deploy. Un modello in produzione interagisce con i dati reali che cambiano nel tempo, portando a un invevitabile degrado delle prestazioni.
In questa fase pratica vedremo come automatizzare l'esportazione e come impostare una mentalità di monitoraggio continuo per garantire la qualità nel tempo.

Mentalità MLOps
Come monitorare la saluta e l'efficienza dell'intelligenza artificiale che interagisce con la realtà
- Il monitoraggio del 'Data Drift' rileva se i dati che arrivano dagli utenti sono diversi da quelli usati durante l'addestramento originale.
- La pipeline di 'Retraining' automatizza l'aggiornamento del modello quando le metriche di accuratezza scendono sotto una soglia critica.
- Il test di inferenza in produzione deve verificare non solo l'accuratezza ma anche il tempo di risposta e il consumo di risorse.
- Il feedback loop permette di raccogliere nuovi dati etichettati per migliorare costantemente l'intelligenza del sistema.
Non vogliamo solo un sisteam intelligente, ma un sistema capace di imparare dai propri errori con i feedback che riceve costantemente.

Rilascio aggiornamenti (la produnza non è mai troppo)
- A/B Testing: consiste nell'inviare parte del traffico al nuovo modello (esempio solo il 5% dei dati al nuovo modello) e parte al vecchio per confrontare le perfomance reali prima del rilascio totale.
- CI/CD per ML: Integrazione e rilascio continui applicati ai modelli, dove ogni miglioramento del codice o dei dati innesca un nuovo deploy verificato.
- Logging e Tracciabilità: salvare ogni predizione e il relativo input consente di effettuare analisi post-mortem in caso di errori macroscopici o bias imprevedibili.

Verso la scalabilità Orizzontale.
Gestione di cluster di modelli
Nelle grandi aziende, non basta più un singolo server, per questo i modelli vengono distribuiti su cluster di server (spesso tramite Kubernetes) per gestire milioni di utenti simultaneamente.
Il modello, in questo scenario, non è più un file, ma un micro-servizio, dinamico ed interconnesso
Comprendere il deploy significa smettere di pensare al modello come un file e iniziare a vederlo come un servizio che è solo una parte di un ingranaggio molto più grande che deve girare in armonia con il resto del software aziendale.


In [ ]:
import tensorflow as tf
import numpy as np
import os

# --- 1. DEFINIZIONE DEL MODELLO (Keras 3 & Functional/Sequential API) ---
# Nel 2025, definire esplicitamente l'Input layer è fondamentale per Keras 3.
# Questo garantisce che il modello sia "tracciabile" per diversi backend (JAX, PyTorch, TF)
# e facilita la conversione in grafi statici come TFLite.

model = tf.keras.Sequential([
    # Definiamo la forma dell'input: un vettore di 10 elementi.
    tf.keras.layers.Input(shape=(10,), name="input_layer"),
    
    # Layer denso con 64 neuroni e attivazione ReLU per apprendere pattern non lineari.
    tf.keras.layers.Dense(64, activation='relu', name="hidden_layer"),
    
    # Layer di output con 10 neuroni (classi) e attivazione Softmax per le probabilità.
    tf.keras.layers.Dense(10, activation='softmax', name="output_layer")  #softmax quindi problema di classificazione
])

# Compilazione: usiamo Adam come ottimizzatore standard e Sparse Categorical Crossentropy
# poiché ci aspettiamo etichette intere (0, 1, 2...) invece di vettori one-hot.
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# --- 2. PROCESSO DI CONVERSIONE IN TENSORFLOW LITE ---
# Carichiamo il modello Keras nel convertitore TFLite.
converter = tf.lite.TFLiteConverter.from_keras_model(model) #convertiamo il modello a tf lite

# Applichiamo la "Dynamic Range Quantization". 
# Questa tecnica riduce il peso dei pesi del modello da float32 a int8 durante il salvataggio,
# diminuendo le dimensioni del file di ~4 volte e velocizzando l'esecuzione senza 
# perdere eccessiva precisione.
converter.optimizations = [tf.lite.Optimize.DEFAULT] #sostituisce i pesi float32 con int8 durante la conversione, riducendone le dimensioni e migliorando le prestazioni su dispositivi mobili/edge

# Generazione del file .tflite (il cuore del modello per dispositivi Edge/Mobile)
tflite_model = converter.convert()  #applico la conversione con le caratteristhce specificate sopra

# Salvataggio fisico su disco.
with open('model_quantized.tflite', 'wb') as f:  #salvataggio modello
    f.write(tflite_model)

print(f"Modello convertito con successo! Dimensione: {len(tflite_model) / 1024:.2f} KB")

# --- 3. INFERENZA MODERNA (Signature Runner) ---
# Invece di manipolare manualmente gli indici dei tensori (metodo legacy), 
# usiamo le 'Signatures', che permettono di chiamare il modello come una funzione Python.

# Inizializziamo l'interprete caricando il file dal disco.
interpreter = tf.lite.Interpreter(model_path="model_quantized.tflite")  #prende il modello nel percorso

# Il Signature Runner mappa automaticamente gli input e gli output tramite i nomi dei layer.
# È il metodo più sicuro e leggibile disponibile nel 2025.
prediction_fn = interpreter.get_signature_runner() #ricostruisce tutta le struttura

# Creiamo un dato di test casuale. Nota: la forma deve essere (batch_size, input_dim), quindi (1, 10).
test_input = np.random.random_sample((1, 10)).astype(np.float32)  #dati di input

# Esecuzione del modello:
# Passiamo l'input usando il nome del layer (spesso 'input_1' o quello definito sopra).
# Il risultato è un dizionario contenente i tensori di output.
output_data = prediction_fn(input_layer=test_input) 

# Estraiamo il risultato dal dizionario usando la chiave corrispondente all'output.
output_key = list(output_data.keys())[0]
probabilities = output_data[output_key]

# Individuiamo la classe con il valore di probabilità più alto.
predicted_class = np.argmax(probabilities)

print(f"Risultato dell'inferenza: Classe {predicted_class}")
print(f"Vettore probabilità: {probabilities}")

INFO:tensorflow:Assets written to: C:\Users\uberti\AppData\Local\Temp\tmpxr4prbgs\assets


INFO:tensorflow:Assets written to: C:\Users\uberti\AppData\Local\Temp\tmpxr4prbgs\assets


Saved artifact at 'C:\Users\uberti\AppData\Local\Temp\tmpxr4prbgs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 10), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1258123556048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1258123557200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1258123554512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1258123554704: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modello convertito con successo! Dimensione: 6.48 KB
Risultato dell'inferenza: Classe 5
Vettore probabilità: [[0.06289959 0.0851652  0.06367753 0.10520791 0.07881629 0.19693096
  0.10191099 0.09550101 0.12074472 0.08914576]]


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
